In [21]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [22]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [23]:
from rag_helper import RAGBase

In [24]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

In [25]:
assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

The word "Olama" doesn't match "Ollama" in our index. We use lexical search, so it looks for the exact word and finds nothing. The LLM gets these bad results and either says "I don't know" or answers with irrelevant information.

This is the limitation of a fixed pipeline. The search runs once with the exact query the user typed, and there's no second chance. The pipeline doesn't know the search failed, so it can't try again with a corrected query.

We need something smarter. We need an agent.

In [26]:
answer = assistant.rag("How do I run Olama locally?")
print(answer)

I couldn’t find any FAQ entry about **Olama** specifically.

The closest relevant guidance is that you **can run the course locally** instead of using Codespaces if you’re comfortable setting up the required tools yourself, such as:

- Python
- `uv`
- Jupyter
- Docker
- any other tools needed for the module

If you run locally, make sure to:

- document your setup
- keep your environment reproducible

If you meant **Ollama** and want course-specific instructions, I don’t have that in the provided context.


## Asking without tools

First, let's see what the LLM does without any tools. We ask it a
course-specific question and look at the answer.

In [27]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'If you just discovered the course, **yes, you may be able to join**—it depends on the course’s enrollment rules and whether registration is still open.\n\nIf you want, I can help you figure it out quickly. Check these:\n- **Enrollment deadline** or add/drop period\n- Whether the course has **prerequisites**\n- If it’s **full** or has a waitlist\n- Whether it’s **self-paced** or part of a scheduled cohort\n\nIf you’d like, send me the course name or a link, and I can help you draft a message to the instructor or admissions office asking if you can still enroll.'

In [28]:
index.search('how to run ollama')

[{'id': '1d0b969028',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'Ollama: How to install Ollama?',
  'answer': 'First, install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\n\n- **macOS**: Download the `.pkg` and install it.\n- **Windows**: Download the `.msi` and install it.\n- **Linux**: Run the following command in the terminal:\n\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\nOnce installed, open a terminal and type:\n\n```bash\nollama run llama3\n```\n\nThis command will:\n\n- Download the LLaMA 3 model (~4GB).\n- Start the model locally.\n- Open a chat-like interface where you can type questions.\n\nTo test the Ollama local server, run the following command:\n\n```bash\ncurl http://localhost:11434\n```\n\nYou should receive a response similar to:\n\n```json\n{"models": [...]}  \n```\n\nThen, install the Python client with:\n\n```bash\npip install ollama\n```\n\n

In [29]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [30]:
search("how do I run ollama?")

[{'id': '1d0b969028',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'Ollama: How to install Ollama?',
  'answer': 'First, install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\n\n- **macOS**: Download the `.pkg` and install it.\n- **Windows**: Download the `.msi` and install it.\n- **Linux**: Run the following command in the terminal:\n\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\nOnce installed, open a terminal and type:\n\n```bash\nollama run llama3\n```\n\nThis command will:\n\n- Download the LLaMA 3 model (~4GB).\n- Start the model locally.\n- Open a chat-like interface where you can type questions.\n\nTo test the Ollama local server, run the following command:\n\n```bash\ncurl http://localhost:11434\n```\n\nYou should receive a response similar to:\n\n```json\n{"models": [...]}  \n```\n\nThen, install the Python client with:\n\n```bash\npip install ollama\n```\n\n

Next we tell the model about this function. The model doesn't see our
Python code, only a schema describing what the function does and what
arguments it takes. LLMs are language agnostic. At the end we're just
making an HTTP call, so we describe the tool in JSON rather than in
Python. The same schema would work from TypeScript or Java.

In [31]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

The `description` is the most important field, because the model reads
it to decide when to call the function. `parameters` is a JSON schema
for the arguments, and we mark `query` as required so the model always
fills it in.

## Sending the question with the tool

Now we send the same question as before, but this time we include the
tool in the request:

In [32]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"join the course discovered course can I join it enrollment eligibility late join"}', call_id='call_4DGNQj6NsCF23sIJ0bB53HmF', name='search', type='function_call', id='fc_0bacc77119123256006aadad7caf4887d2a5df09d7dff95655', async_=None, caller=None, namespace=None, status='completed')]

Look at the output. Instead of a message with the answer, the response
contains a `function_call` entry. The model decided it needs to search
the FAQ before answering. Rather than reply, it asked us to run the
search function first.

Look at the arguments too. The model didn't pass our question
verbatim. It judged the raw question wasn't the best query to search
with. So it rewrote our enrollment question into search keywords like
"enroll late join course".

## Executing the function and sending the result back

The function call contains JSON arguments. We parse them, call our
`search` function, and serialize the result.

In [33]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [34]:
print(result_json)

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "5cc511f85b",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "Does the course certificate show the number of course hours?",
    "answer": "No. The certificate does not state a total number of hours."
  },
  {
    "id": "04919992b3",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "How should I start the course and follow the weekly workflow?",
    "answer": "Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomc

Now we send this result back to the model.\
First, we add the model's output to the conversation history - the model needs to see its own\
function call.\
Then we add the tool result.

In [35]:
function_call_output = {
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
}

We cannot simply make a nother call to LLM with this output, we have to include the history from the first call\
because LLMs are stateless in nature. 

In [36]:
messages.extend(response.output)

In [37]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join the course discovered course can I join it enrollment eligibility late join"}', call_id='call_4DGNQj6NsCF23sIJ0bB53HmF', name='search', type='function_call', id='fc_0bacc77119123256006aadad7caf4887d2a5df09d7dff95655', async_=None, caller=None, namespace=None, status='completed')]

In [38]:
messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join the course discovered course can I join it enrollment eligibility late join"}', call_id='call_4DGNQj6NsCF23sIJ0bB53HmF', name='search', type='function_call', id='fc_0bacc77119123256006aadad7caf4887d2a5df09d7dff95655', async_=None, caller=None, namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_4DGNQj6NsCF23sIJ0bB53HmF',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "5cc511f85b",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Does the course certificate show 

The `call_id` links the tool result to the specific function call the
model requested. If the model makes multiple function calls in one
turn, each one gets its own `call_id`.

## Asking the model again

We call the API a second time with the expanded history:

In [39]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes — you can still join and start learning anytime.\n\nIf you want a certificate, you’ll need to submit your project while the course is still accepting submissions.'

This time the model has the original question, its own decision to
call `search`, and the FAQ results. It can now produce a proper
course-specific answer.

We have to send the whole history because LLMs are stateless between
API calls. The memory is the list you send as `input`. If you send
only the tool result, the model has no idea what's going on. So on
this second call we replay everything we have so far. That means the
question, the decision to call `search`, and the result we got back.

That's the full function-calling loop for a single turn. With plain
RAG we made one call, and here we make two. Turning RAG agentic means
more round-trips.

People call this pattern "agentic RAG", "tool use", or "function
calling". The idea behind all of them is the same. The LLM decides
which tools to call.

## Token usage and cost

We just made two API calls instead of one. Each call we send to the
model costs money, so it's worth checking how much one tool-using turn
actually costs.

The response has a `usage` field with the token counts:

In [40]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(786, 36)

For each model the provider publishes a price per million input tokens
and per million output tokens. Plug those numbers in to convert tokens
to dollars.

In [ ]:
def calculate_gpt54mini_price(input_tokens, output_tokens): 
    INPUT_PRICE_PER_MILLION = 0.75  # prices for GPT-5.4-mini for 18th Dec 2026
    OUTPUT_PRICE_PER_MILLION = 4.5

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(786, 36)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0007515


This usage is only for the second API call. The first call also has
its own usage and its own cost. That was the call where the model
decided to invoke `search`. Two calls means we pay twice. We pay even
more on the second call, because we resend the full history as input.

With a real agent loop the model can make many calls, so the costs add
up. Keep an eye on `usage` while you develop.

In [44]:
def calculate_agent_loop_cost(
    first_call_input,
    first_call_output,
    second_call_cached_input,
    second_call_new_input,
    second_call_output,
):
    """
    Calculate GPT-5.4 mini API cost for a two-call agent loop.

    Prices per 1M tokens:
        Input:        $0.75
        Cached input: $0.075
        Output:       $4.50
    """

    INPUT_PRICE = 0.75
    CACHED_INPUT_PRICE = 0.075
    OUTPUT_PRICE = 4.50

    # -------------------------
    # FIRST CALL
    # -------------------------

    first_input_cost = (
        first_call_input / 1_000_000
    ) * INPUT_PRICE

    first_output_cost = (
        first_call_output / 1_000_000
    ) * OUTPUT_PRICE

    first_call_total = (
        first_input_cost +
        first_output_cost
    )

    # -------------------------
    # SECOND CALL
    # -------------------------

    second_cached_input_cost = (
        second_call_cached_input / 1_000_000
    ) * CACHED_INPUT_PRICE

    second_new_input_cost = (
        second_call_new_input / 1_000_000
    ) * INPUT_PRICE

    second_output_cost = (
        second_call_output / 1_000_000
    ) * OUTPUT_PRICE

    second_call_total = (
        second_cached_input_cost +
        second_new_input_cost +
        second_output_cost
    )

    # -------------------------
    # TOTAL
    # -------------------------

    total_cost = (
        first_call_total +
        second_call_total
    )

    return {
        "first_call": {
            "input_cost": first_input_cost,
            "output_cost": first_output_cost,
            "total": first_call_total,
        },
        "second_call": {
            "cached_input_cost": second_cached_input_cost,
            "new_input_cost": second_new_input_cost,
            "output_cost": second_output_cost,
            "total": second_call_total,
        },
        "total_agent_cost": total_cost,
    }

In [47]:
result = calculate_agent_loop_cost(
    first_call_input=786,
    first_call_output=36,
    second_call_cached_input=786,
    second_call_new_input=2500,
    second_call_output=400,
)

print(f"First call:  ${result['first_call']['total']:.9f}")
print(f"Second call: ${result['second_call']['total']:.9f}")
print(f"Total:       ${result['total_agent_cost']:.9f}")

First call:  $0.000751500
Second call: $0.003733950
Total:       $0.004485450
